# V6.2 — Rebalance Train/Val, Keep Test Exactly Fixed

This is the correction step after V6.1.

## Why

V6.1 passed integrity checks, but `bird_nest` was severely train-heavy:

- train: 6,127 images / 6,854 boxes
- val: 118 images / 128 boxes
- test: 167 images / 178 boxes

The previous `freeze_ready=True` meant **data integrity passed**. It did **not** mean the class distribution was statistically suitable for benchmarking.

## Policy for V6.2

- **TEST is immutable.**
- Only the union of current TRAIN + VAL is re-split.
- Keep the same overall split sizes:
  - train = 17,655
  - val = 2,263
  - test = 2,012
- Use multilabel stratification over:
  - the six detection classes;
  - source (`base` vs `FOTL`) as additional stratification indicators.
- Preserve exactly the same image pool.
- Do not add or remove images.
- Verify no SHA256 duplicate crosses train/val/test.
- Export only a **split manifest + freeze record**, not another 20+ GB dataset copy.

Result: V6.2 is a new frozen split definition built on the same V6.1 image pool.


In [ ]:
from pathlib import Path
import os, sys, json, csv, shutil, subprocess, hashlib, zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone

W = Path("/kaggle/working")
TMP = Path("/tmp/v6p2_resplit")
BASE_EXTRACT = TMP/"base"
OVERLAY_EXTRACT = TMP/"overlay"
OLD_RUNTIME = TMP/"v6p1_runtime"
NEW_RUNTIME = TMP/"v6p2_runtime"
REPORTS = W/"v6p2_freeze_records"

for p in [TMP, BASE_EXTRACT, OVERLAY_EXTRACT, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

V6P1_TAG = "v6p1_5d0b1149891c"
BASE_TAG = "56b7ea194ee5"
OVERLAY_TAG = "ee2d3c4497fd"

DATA_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets"
V6P1_ROOT = f"{DATA_ROOT}/v6p1/{V6P1_TAG}"

BASE_STASH = f"{DATA_ROOT}/curated_v6_rs1280_{BASE_TAG}.tar"
OVERLAY_STASH = f"{V6P1_ROOT}/v6p1_fotl_overlay_{OVERLAY_TAG}.tar.gz"
FREEZE_STASH = f"{V6P1_ROOT}/v6p1.freeze.json"

EXPECTED_NAMES = [
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand",
]

EXPECTED_SPLITS = {"train":17655, "val":2263, "test":2012}
SEED = 42

# New version path
V6P2_ROOT = f"{DATA_ROOT}/v6p2"
UPLOAD_TO_CAMBER = True


In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pyyaml", "pandas", "iterative-stratification"
])

import numpy as np
import pandas as pd
import yaml
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

def images_under(root):
    return sorted(
        p for p in Path(root).rglob("*")
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    )

def sha256_file(path, chunk=4*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()

def utc_now():
    return datetime.now(timezone.utc).isoformat()


## 1. Restore exact V6.1 artifacts


In [ ]:
from kaggle_secrets import UserSecretsClient

key=UserSecretsClient().get_secret("CAMBER_API_KEY")
if not key:
    raise RuntimeError("Missing CAMBER_API_KEY")

os.environ["CAMBER_API_KEY"]=key

if shutil.which("camber") is None and not (Path.home()/".camber/bin/camber").exists():
    subprocess.check_call("curl -sL https://cli.cambercloud.com/install-v2.sh | bash",shell=True)

os.environ["PATH"]=f"{Path.home()}/.camber/bin:{Path.home()}/.local/bin:"+os.environ.get("PATH","")
CAMBER=shutil.which("camber") or str(Path.home()/".camber/bin/camber")

def stash_cp(src,dst,required=True):
    dst=Path(dst); dst.parent.mkdir(parents=True,exist_ok=True)
    r=subprocess.run([CAMBER,"stash","cp",str(src),str(dst)],
                     text=True,capture_output=True,env=os.environ.copy())
    ok=r.returncode==0 and dst.exists()
    if required and not ok:
        raise RuntimeError(r.stderr[-1800:] or r.stdout[-1800:])
    return ok

def stash_put(src,dst):
    src=Path(src)
    r=subprocess.run([CAMBER,"stash","cp",str(src),str(dst)],
                     text=True,capture_output=True,env=os.environ.copy())
    if r.returncode!=0:
        raise RuntimeError(r.stderr[-1800:] or r.stdout[-1800:])
    print("Uploaded:",dst)

freeze_local=TMP/"v6p1.freeze.json"
stash_cp(FREEZE_STASH,freeze_local)
v6p1=json.loads(freeze_local.read_text())

assert v6p1["final_tag"]==V6P1_TAG
assert v6p1["freeze_ready"] is True
assert v6p1["final_split_counts"]==EXPECTED_SPLITS

base_tar=TMP/f"base_{BASE_TAG}.tar"
overlay_tar=TMP/f"overlay_{OVERLAY_TAG}.tar.gz"

if not any(BASE_EXTRACT.rglob("data.yaml")):
    stash_cp(BASE_STASH,base_tar)
    subprocess.check_call(["tar","-xf",str(base_tar),"-C",str(BASE_EXTRACT)])
    base_tar.unlink(missing_ok=True)

if not any(OVERLAY_EXTRACT.rglob("data.yaml")):
    stash_cp(OVERLAY_STASH,overlay_tar)
    subprocess.check_call(["tar","-xzf",str(overlay_tar),"-C",str(OVERLAY_EXTRACT)])
    overlay_tar.unlink(missing_ok=True)

def normalize_names(names):
    if isinstance(names,dict):
        return [names[k] for k in sorted(names,key=lambda x:int(x))]
    return list(names)

def find_root(root):
    for yp in Path(root).rglob("data.yaml"):
        try:
            cfg=yaml.safe_load(yp.read_text())
            if normalize_names(cfg.get("names",[]))==EXPECTED_NAMES:
                return yp.parent
        except Exception:
            pass
    raise RuntimeError(f"No matching dataset under {root}")

BASE=find_root(BASE_EXTRACT)
OVERLAY=find_root(OVERLAY_EXTRACT)

print("BASE:",BASE)
print("OVERLAY:",OVERLAY)


## 2. Reconstruct the exact V6.1 runtime union


In [ ]:
if OLD_RUNTIME.exists():
    shutil.rmtree(OLD_RUNTIME)

for sp in ["train","val","test"]:
    for src in ["base","fotl"]:
        (OLD_RUNTIME/"images"/sp/src).mkdir(parents=True,exist_ok=True)
        (OLD_RUNTIME/"labels"/sp/src).mkdir(parents=True,exist_ok=True)

def link_source(src_root, split, source_name, runtime_root):
    iroot=src_root/"images"/split
    lroot=src_root/"labels"/split

    for ip in images_under(iroot):
        rel=ip.relative_to(iroot)
        lp=lroot/rel.with_suffix(".txt")

        dip=runtime_root/"images"/split/source_name/rel
        dlp=runtime_root/"labels"/split/source_name/rel.with_suffix(".txt")

        dip.parent.mkdir(parents=True,exist_ok=True)
        dlp.parent.mkdir(parents=True,exist_ok=True)

        os.symlink(ip,dip)
        if lp.exists():
            os.symlink(lp,dlp)
        else:
            dlp.write_text("")

for sp in ["train","val","test"]:
    link_source(BASE,sp,"base",OLD_RUNTIME)
    link_source(OVERLAY,sp,"fotl",OLD_RUNTIME)

counts={sp:len(images_under(OLD_RUNTIME/"images"/sp)) for sp in ["train","val","test"]}
print(counts)
assert counts==EXPECTED_SPLITS


## 3. Build authoritative image records

Each image record contains:
- original V6.1 split;
- source (`base` / `fotl`);
- class-presence vector;
- box counts;
- SHA256.

TEST records are marked immutable.


In [ ]:
records=[]

for sp in ["train","val","test"]:
    iroot=OLD_RUNTIME/"images"/sp
    lroot=OLD_RUNTIME/"labels"/sp

    for ip in images_under(iroot):
        rel=ip.relative_to(iroot)
        source=rel.parts[0]
        lp=lroot/rel.with_suffix(".txt")

        box_counts=[0]*len(EXPECTED_NAMES)
        if lp.exists():
            for line in lp.read_text(errors="ignore").splitlines():
                if not line.strip(): continue
                parts=line.split()
                cid=int(float(parts[0]))
                vals=list(map(float,parts[1:5]))
                assert 0 <= cid < len(EXPECTED_NAMES)
                assert all(0 <= x <= 1 for x in vals)
                box_counts[cid]+=1

        records.append({
            "old_split":sp,
            "source":source,
            "relpath":rel.as_posix(),
            "image_path":str(ip),
            "label_path":str(lp),
            "sha256":sha256_file(ip),
            "class_presence":[int(x>0) for x in box_counts],
            "box_counts":box_counts,
        })

df=pd.DataFrame(records)
print(df.groupby(["old_split","source"]).size())
print("Total:",len(df))
assert len(df)==sum(EXPECTED_SPLITS.values())


## 4. Immutable TEST gate

TEST will not be touched.

The exact ordered set of TEST SHA256 values is saved before re-splitting.


In [ ]:
test_records=[r for r in records if r["old_split"]=="test"]
pool_records=[r for r in records if r["old_split"] in ("train","val")]

OLD_TEST_HASHES=sorted(r["sha256"] for r in test_records)
OLD_POOL_HASHES=sorted(r["sha256"] for r in pool_records)

assert len(test_records)==EXPECTED_SPLITS["test"]
assert len(pool_records)==EXPECTED_SPLITS["train"]+EXPECTED_SPLITS["val"]

print("Immutable test images:",len(test_records))
print("Train+val pool:",len(pool_records))


## 5. Re-split TRAIN + VAL only

Stratification target includes:
- six class-presence columns;
- source_base;
- source_fotl.

This keeps source proportions from drifting too much while correcting class imbalance.

The validation size remains exactly 2,263 images.


In [ ]:
Y=[]

for r in pool_records:
    class_bits=list(r["class_presence"])
    source_bits=[
        int(r["source"]=="base"),
        int(r["source"]=="fotl"),
    ]
    Y.append(class_bits+source_bits)

Y=np.asarray(Y,dtype=np.int64)
X=np.arange(len(pool_records)).reshape(-1,1)

val_fraction=EXPECTED_SPLITS["val"]/len(pool_records)

splitter=MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=val_fraction,
    random_state=SEED,
)

idx_train,idx_val=next(splitter.split(X,Y))

new_train=[pool_records[int(i)] for i in idx_train]
new_val=[pool_records[int(i)] for i in idx_val]
new_test=test_records

print("new train:",len(new_train))
print("new val:",len(new_val))
print("new test:",len(new_test))

assert len(new_train)==EXPECTED_SPLITS["train"]
assert len(new_val)==EXPECTED_SPLITS["val"]
assert len(new_test)==EXPECTED_SPLITS["test"]


## 6. Verify test unchanged and image pool unchanged


In [ ]:
NEW_TEST_HASHES=sorted(r["sha256"] for r in new_test)
NEW_POOL_HASHES=sorted(r["sha256"] for r in new_train+new_val)

assert NEW_TEST_HASHES==OLD_TEST_HASHES, "TEST changed!"
assert NEW_POOL_HASHES==OLD_POOL_HASHES, "Train+val image pool changed!"

train_hash=set(r["sha256"] for r in new_train)
val_hash=set(r["sha256"] for r in new_val)
test_hash=set(r["sha256"] for r in new_test)

cross_tv=train_hash & val_hash
cross_tt=train_hash & test_hash
cross_vt=val_hash & test_hash

print("Exact duplicate overlap train-val:",len(cross_tv))
print("Exact duplicate overlap train-test:",len(cross_tt))
print("Exact duplicate overlap val-test:",len(cross_vt))

assert not cross_tv
assert not cross_tt
assert not cross_vt

print("✅ Test immutable")
print("✅ Same train+val pool")
print("✅ No exact SHA256 leakage")


## 7. Compare V6.1 vs V6.2 distribution

This is the decision table.

We specifically check whether `bird_nest` validation support is now representative.


In [ ]:
def summarize(recs):
    img=Counter()
    box=Counter()
    for r in recs:
        for cid,p in enumerate(r["class_presence"]):
            if p:
                img[cid]+=1
        for cid,n in enumerate(r["box_counts"]):
            box[cid]+=n
    return img,box

old_summary={}
for sp in ["train","val","test"]:
    old_summary[sp]=summarize([r for r in records if r["old_split"]==sp])

new_summary={
    "train":summarize(new_train),
    "val":summarize(new_val),
    "test":summarize(new_test),
}

rows=[]
for cid,name in enumerate(EXPECTED_NAMES):
    for sp in ["train","val","test"]:
        old_img,old_box=old_summary[sp]
        new_img,new_box=new_summary[sp]
        rows.append({
            "class_id":cid,
            "class_name":name,
            "split":sp,
            "v6p1_images":old_img[cid],
            "v6p2_images":new_img[cid],
            "v6p1_boxes":old_box[cid],
            "v6p2_boxes":new_box[cid],
        })

cmp=pd.DataFrame(rows)
display(cmp)
cmp.to_csv(REPORTS/"v6p1_vs_v6p2_distribution.csv",index=False)

bird=cmp[cmp.class_name=="bird_nest"]
display(bird)


In [ ]:
# Validation representation gate.
# Since TEST is fixed, use train+val prevalence as the target.
gates=[]

for cid,name in enumerate(EXPECTED_NAMES):
    train_img=new_summary["train"][0][cid]
    val_img=new_summary["val"][0][cid]
    pool_img=train_img+val_img

    train_box=new_summary["train"][1][cid]
    val_box=new_summary["val"][1][cid]
    pool_box=train_box+val_box

    val_img_ratio=val_img/pool_img if pool_img else 0
    val_box_ratio=val_box/pool_box if pool_box else 0

    gates.append({
        "class_id":cid,
        "class_name":name,
        "train_images":train_img,
        "val_images":val_img,
        "val_image_ratio_within_trainval":val_img_ratio,
        "train_boxes":train_box,
        "val_boxes":val_box,
        "val_box_ratio_within_trainval":val_box_ratio,
    })

gate_df=pd.DataFrame(gates)
display(gate_df)
gate_df.to_csv(REPORTS/"v6p2_trainval_balance.csv",index=False)

target_ratio=EXPECTED_SPLITS["val"]/(EXPECTED_SPLITS["train"]+EXPECTED_SPLITS["val"])
print("Global val ratio within train+val:",target_ratio)

bird_row=gate_df[gate_df.class_name=="bird_nest"].iloc[0]
assert bird_row["val_image_ratio_within_trainval"] >= 0.08, (
    "bird_nest val representation still too low"
)

print("✅ bird_nest validation representation gate passed.")


## 8. Build V6.2 runtime split using symlinks

Same image bytes, only the split assignment changes.


In [ ]:
if NEW_RUNTIME.exists():
    shutil.rmtree(NEW_RUNTIME)

for sp in ["train","val","test"]:
    (NEW_RUNTIME/"images"/sp).mkdir(parents=True,exist_ok=True)
    (NEW_RUNTIME/"labels"/sp).mkdir(parents=True,exist_ok=True)

def link_record(r,sp):
    rel=Path(r["relpath"])
    dip=NEW_RUNTIME/"images"/sp/rel
    dlp=NEW_RUNTIME/"labels"/sp/rel.with_suffix(".txt")

    dip.parent.mkdir(parents=True,exist_ok=True)
    dlp.parent.mkdir(parents=True,exist_ok=True)

    os.symlink(r["image_path"],dip)

    lp=Path(r["label_path"])
    if lp.exists():
        os.symlink(lp,dlp)
    else:
        dlp.write_text("")

for r in new_train: link_record(r,"train")
for r in new_val: link_record(r,"val")
for r in new_test: link_record(r,"test")

runtime_counts={
    sp:len(images_under(NEW_RUNTIME/"images"/sp))
    for sp in ["train","val","test"]
}
print(runtime_counts)
assert runtime_counts==EXPECTED_SPLITS

(NEW_RUNTIME/"data.yaml").write_text(yaml.safe_dump({
    "path":str(NEW_RUNTIME),
    "train":"images/train",
    "val":"images/val",
    "test":"images/test",
    "names":EXPECTED_NAMES,
},sort_keys=False))


## 9. Save split manifest and freeze V6.2

The manifest is sufficient to reconstruct V6.2 from V6.1 artifacts.


In [ ]:
manifest=[]

for sp,recs in [("train",new_train),("val",new_val),("test",new_test)]:
    for r in recs:
        manifest.append({
            "split":sp,
            "source":r["source"],
            "relpath":r["relpath"],
            "sha256":r["sha256"],
            "previous_v6p1_split":r["old_split"],
        })

manifest_df=pd.DataFrame(manifest).sort_values(["split","source","relpath"])
manifest_path=REPORTS/"v6p2_split_manifest.csv"
manifest_df.to_csv(manifest_path,index=False)

h=hashlib.sha256()
h.update(V6P1_TAG.encode())
h.update(json.dumps(EXPECTED_NAMES).encode())
for row in manifest_df.to_dict("records"):
    h.update(json.dumps(row,sort_keys=True).encode())

SPLIT_SHA=h.hexdigest()
V6P2_TAG=f"v6p2_{SPLIT_SHA[:12]}"

freeze={
    "schema_version":"v6p2-split-freeze-1",
    "created_utc":utc_now(),
    "dataset_name":"V6.2",
    "final_tag":V6P2_TAG,
    "split_sha256":SPLIT_SHA,
    "parent_dataset_tag":V6P1_TAG,
    "base_tag":BASE_TAG,
    "overlay_tag":OVERLAY_TAG,
    "taxonomy":EXPECTED_NAMES,
    "split_policy":{
        "test":"exactly preserved from V6.1",
        "train_val":"re-split only within the V6.1 train+val image pool",
        "method":"MultilabelStratifiedShuffleSplit",
        "stratification_features":"six class-presence labels + source_base + source_fotl",
        "seed":SEED,
        "train_count":EXPECTED_SPLITS["train"],
        "val_count":EXPECTED_SPLITS["val"],
        "test_count":EXPECTED_SPLITS["test"],
    },
    "integrity":{
        "test_sha256_set_unchanged":NEW_TEST_HASHES==OLD_TEST_HASHES,
        "trainval_sha256_multiset_unchanged":NEW_POOL_HASHES==OLD_POOL_HASHES,
        "exact_train_val_overlap":len(cross_tv),
        "exact_train_test_overlap":len(cross_tt),
        "exact_val_test_overlap":len(cross_vt),
    },
    "freeze_ready":True,
}

freeze_path=REPORTS/"v6p2.freeze.json"
freeze_path.write_text(json.dumps(freeze,indent=2))

print(json.dumps(freeze,indent=2))


## 10. Human-readable report


In [ ]:
final_rows=[]
for cid,name in enumerate(EXPECTED_NAMES):
    total_boxes=0
    total_images=0
    row={"class":name}

    for sp in ["train","val","test"]:
        img=new_summary[sp][0][cid]
        box=new_summary[sp][1][cid]
        row[f"{sp}_images"]=img
        row[f"{sp}_boxes"]=box
        total_images += img
        total_boxes += box

    row["total_images_with_class"]=total_images
    row["total_boxes"]=total_boxes
    final_rows.append(row)

final_df=pd.DataFrame(final_rows)
final_df.to_csv(REPORTS/"v6p2_final_distribution.csv",index=False)
display(final_df)

report=[]
report.append("# V6.2 Dataset Split Report")
report.append("")
report.append(f"- Final tag: `{V6P2_TAG}`")
report.append(f"- Parent: `{V6P1_TAG}`")
report.append("- TEST: unchanged from V6.1")
report.append("- TRAIN+VAL image pool: unchanged")
report.append(f"- Train: {EXPECTED_SPLITS['train']}")
report.append(f"- Validation: {EXPECTED_SPLITS['val']}")
report.append(f"- Test: {EXPECTED_SPLITS['test']}")
report.append("")
report.append("## Final distribution")
report.append("")
report.append(final_df.to_markdown(index=False))
report.append("")
report.append("## Methodological note")
report.append(
    "V6.1 passed integrity and duplicate checks, but its bird_nest validation "
    "representation was too small relative to the training data. V6.2 therefore "
    "keeps the test set exactly fixed and re-stratifies only the original train "
    "and validation pool, preserving total split sizes and source composition."
)

report_path=REPORTS/"V6P2_DATASET_REPORT.md"
report_path.write_text("\n".join(report))
print(report_path.read_text()[:6000])


## 11. Upload V6.2 freeze records to Camber

Only split records are uploaded. Image artifacts remain the same V6.1 base + FOTL overlay.


In [ ]:
records_zip=W/f"{V6P2_TAG}_freeze_records.zip"

with zipfile.ZipFile(records_zip,"w",zipfile.ZIP_DEFLATED) as z:
    for p in REPORTS.rglob("*"):
        if p.is_file():
            z.write(p,p.relative_to(REPORTS.parent))

if UPLOAD_TO_CAMBER:
    remote=f"{V6P2_ROOT}/{V6P2_TAG}"

    for p in [
        freeze_path,
        manifest_path,
        REPORTS/"v6p1_vs_v6p2_distribution.csv",
        REPORTS/"v6p2_trainval_balance.csv",
        REPORTS/"v6p2_final_distribution.csv",
        report_path,
        records_zip,
    ]:
        stash_put(p,f"{remote}/{p.name}")

    print("✅ FINAL DATASET TAG:",V6P2_TAG)
    print("✅ TEST preserved exactly from:",V6P1_TAG)
    print("✅ CAMBER ROOT:",remote)
else:
    print("FINAL DATASET TAG:",V6P2_TAG)


# After this notebook

If the final `bird_nest` validation count is now healthy and the notebook prints:

```text
✅ FINAL DATASET TAG: v6p2_...
```

then **V6.2 replaces V6.1 for all future baseline/ablation training**.

The TEST set is still the exact same held-out set from V6.1, so previous test integrity is preserved.

Do not train B0-B3 on V6.1 after this correction.
